In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go

# ============================================================
# ChatGPT Event Pre/Post Connectedness Comparison
# - input: tvpvar_input_scaled.csv, gfevd_all.npy
# - output: interactive figures in notebook / VS Code
# ============================================================

# -----------------------------
# 1) Config
# -----------------------------
BASE_DIR = Path("./")

INPUT_FILE = BASE_DIR / "tvpvar_input_scaled.csv"
GFEVD_FILE = BASE_DIR / "gfevd_all.npy"

VAR_NAMES = [
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
    "dlog_DXY",
    "d_UST10Y",
    "d_VIX"
]

PRE_START  = "2022-05-01"
PRE_END    = "2022-10-31"
POST_START = "2023-01-01"
POST_END   = "2023-06-30"

# -----------------------------
# 2) Load
# -----------------------------
df = pd.read_csv(INPUT_FILE)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

gfevd_all = np.load(GFEVD_FILE)

if gfevd_all.ndim != 3:
    raise ValueError("gfevd_all.npy must have shape (T, k, k)")

if gfevd_all.shape[1] != len(VAR_NAMES):
    raise ValueError(
        f"VAR_NAMES length ({len(VAR_NAMES)}) != gfevd dimension ({gfevd_all.shape[1]})"
    )

# -----------------------------
# 3) Align dates
# -----------------------------
dates = df["Date"].copy()

diff = len(dates) - gfevd_all.shape[0]
if diff > 0:
    dates = dates.iloc[diff:].reset_index(drop=True)
elif diff < 0:
    raise ValueError(
        f"Date rows ({len(dates)}) < gfevd rows ({gfevd_all.shape[0]})"
    )

if len(dates) != gfevd_all.shape[0]:
    raise ValueError("Date alignment failed")

# -----------------------------
# 4) Functions
# -----------------------------
def compute_tci(g: np.ndarray) -> np.ndarray:
    k = g.shape[1]
    diag = np.einsum("tii->t", g)
    offdiag = g.sum(axis=(1, 2)) - diag
    return offdiag / k * 100.0

def compute_net(g: np.ndarray) -> np.ndarray:
    """
    NET_t(i) = TO_t(i) - FROM_t(i)
    """
    T, k, _ = g.shape
    net = np.zeros((T, k))
    for t in range(T):
        m = g[t] * 100.0
        diag = np.diag(m)
        from_others = m.sum(axis=1) - diag
        to_others = m.sum(axis=0) - diag
        net[t] = to_others - from_others
    return net

def mean_connectedness_matrix(g: np.ndarray) -> np.ndarray:
    return g.mean(axis=0) * 100.0

def period_mask(date_series: pd.Series, start: str, end: str) -> pd.Series:
    return (date_series >= pd.to_datetime(start)) & (date_series <= pd.to_datetime(end))

# -----------------------------
# 5) Slice pre/post
# -----------------------------
pre_mask = period_mask(dates, PRE_START, PRE_END)
post_mask = period_mask(dates, POST_START, POST_END)

if pre_mask.sum() == 0:
    raise ValueError("No observations in pre period")
if post_mask.sum() == 0:
    raise ValueError("No observations in post period")

pre_dates = dates[pre_mask].reset_index(drop=True)
post_dates = dates[post_mask].reset_index(drop=True)

pre_g = gfevd_all[pre_mask.values]
post_g = gfevd_all[post_mask.values]

print(f"[PRE ] {PRE_START} ~ {PRE_END}  -> {len(pre_dates)} obs")
print(f"[POST] {POST_START} ~ {POST_END} -> {len(post_dates)} obs")

# -----------------------------
# 6) Compute pre/post
# -----------------------------
pre_tci = compute_tci(pre_g)
post_tci = compute_tci(post_g)

pre_net = compute_net(pre_g)
post_net = compute_net(post_g)

pre_mat = mean_connectedness_matrix(pre_g)
post_mat = mean_connectedness_matrix(post_g)

delta_mat = post_mat - pre_mat
delta_tci = float(post_tci.mean() - pre_tci.mean())
delta_net = post_net.mean(axis=0) - pre_net.mean(axis=0)

# -----------------------------
# 7) Summary tables
# -----------------------------
summary_df = pd.DataFrame([
    {
        "period": "pre",
        "start": PRE_START,
        "end": PRE_END,
        "n_obs": len(pre_dates),
        "tci_mean": float(pre_tci.mean()),
        "tci_std": float(pre_tci.std(ddof=1)) if len(pre_tci) > 1 else np.nan,
        "tci_min": float(pre_tci.min()),
        "tci_max": float(pre_tci.max())
    },
    {
        "period": "post",
        "start": POST_START,
        "end": POST_END,
        "n_obs": len(post_dates),
        "tci_mean": float(post_tci.mean()),
        "tci_std": float(post_tci.std(ddof=1)) if len(post_tci) > 1 else np.nan,
        "tci_min": float(post_tci.min()),
        "tci_max": float(post_tci.max())
    },
    {
        "period": "delta(post-pre)",
        "start": "",
        "end": "",
        "n_obs": np.nan,
        "tci_mean": delta_tci,
        "tci_std": np.nan,
        "tci_min": np.nan,
        "tci_max": np.nan
    }
])

net_mean_df = pd.DataFrame({
    "Variable": VAR_NAMES,
    "NET_mean_pre": pre_net.mean(axis=0),
    "NET_mean_post": post_net.mean(axis=0),
    "Delta_NET": delta_net
})

delta_conn_df = pd.DataFrame(delta_mat, index=VAR_NAMES, columns=VAR_NAMES)

print("\n[TCI summary]")
print(summary_df)

print("\n[NET mean summary]")
print(net_mean_df)

print("\n[Delta Connectedness: Post - Pre]")
print(delta_conn_df.round(2))

# -----------------------------
# 8) TCI figure
# -----------------------------
fig_tci = go.Figure()

fig_tci.add_trace(
    go.Scatter(
        x=pre_dates,
        y=pre_tci,
        mode="lines",
        name="Pre TCI"
    )
)

fig_tci.add_trace(
    go.Scatter(
        x=post_dates,
        y=post_tci,
        mode="lines",
        name="Post TCI"
    )
)

fig_tci.add_hline(
    y=float(pre_tci.mean()),
    line_width=1,
    line_dash="dot",
    annotation_text=f"Pre mean={pre_tci.mean():.2f}",
    annotation_position="top left"
)

fig_tci.add_hline(
    y=float(post_tci.mean()),
    line_width=1,
    line_dash="dot",
    annotation_text=f"Post mean={post_tci.mean():.2f}",
    annotation_position="top right"
)

fig_tci.update_layout(
    title="ChatGPT Event: Pre vs Post TCI",
    xaxis_title="Date",
    yaxis_title="TCI",
    height=600,
    hovermode="x unified"
)

fig_tci.show()

# -----------------------------
# 9) NET figure
# -----------------------------
fig_net = go.Figure()

for i, var in enumerate(VAR_NAMES):
    fig_net.add_trace(
        go.Scatter(
            x=pre_dates,
            y=pre_net[:, i],
            mode="lines",
            name=f"Pre {var}"
        )
    )

for i, var in enumerate(VAR_NAMES):
    fig_net.add_trace(
        go.Scatter(
            x=post_dates,
            y=post_net[:, i],
            mode="lines",
            name=f"Post {var}"
        )
    )

fig_net.add_hline(y=0, line_width=1, line_color="black")

fig_net.update_layout(
    title="ChatGPT Event: Pre vs Post NET Spillover",
    xaxis_title="Date",
    yaxis_title="NET Spillover",
    height=700,
    hovermode="x unified"
)

fig_net.show()

# -----------------------------
# 10) Heatmap: Pre / Post / Delta
# -----------------------------
fig_heat = go.Figure()

fig_heat.add_trace(
    go.Heatmap(
        z=pre_mat,
        x=VAR_NAMES,
        y=VAR_NAMES,
        text=np.round(pre_mat, 1),
        texttemplate="%{text}",
        textfont={"size": 10},
        visible=True,
        colorbar=dict(title="Connectedness (%)"),
        hovertemplate="Receiver=%{y}<br>Transmitter=%{x}<br>Value=%{z:.2f}<extra></extra>"
    )
)

fig_heat.add_trace(
    go.Heatmap(
        z=post_mat,
        x=VAR_NAMES,
        y=VAR_NAMES,
        text=np.round(post_mat, 1),
        texttemplate="%{text}",
        textfont={"size": 10},
        visible=False,
        colorbar=dict(title="Connectedness (%)"),
        hovertemplate="Receiver=%{y}<br>Transmitter=%{x}<br>Value=%{z:.2f}<extra></extra>"
    )
)

fig_heat.add_trace(
    go.Heatmap(
        z=delta_mat,
        x=VAR_NAMES,
        y=VAR_NAMES,
        text=np.round(delta_mat, 1),
        texttemplate="%{text}",
        textfont={"size": 10},
        visible=False,
        colorscale="RdBu",
        zmid=0,
        colorbar=dict(title="Delta (%)"),
        hovertemplate="Receiver=%{y}<br>Transmitter=%{x}<br>Delta=%{z:.2f}<extra></extra>"
    )
)

fig_heat.update_layout(
    title="Pre Mean Connectedness Heatmap",
    xaxis_title="Transmitter",
    yaxis_title="Receiver",
    height=720,
    updatemenus=[
        dict(
            buttons=[
                dict(
                    label="Pre",
                    method="update",
                    args=[
                        {"visible": [True, False, False]},
                        {"title": "Pre Mean Connectedness Heatmap"}
                    ]
                ),
                dict(
                    label="Post",
                    method="update",
                    args=[
                        {"visible": [False, True, False]},
                        {"title": "Post Mean Connectedness Heatmap"}
                    ]
                ),
                dict(
                    label="Delta (Post - Pre)",
                    method="update",
                    args=[
                        {"visible": [False, False, True]},
                        {"title": "Delta Connectedness Heatmap (Post - Pre)"}
                    ]
                ),
            ],
            direction="down",
            x=1.02,
            y=1.0,
            xanchor="left",
            yanchor="top"
        )
    ]
)

fig_heat.show()

# -----------------------------
# 11) Delta NET bar chart
# -----------------------------
fig_delta_net = go.Figure()

fig_delta_net.add_trace(
    go.Bar(
        x=VAR_NAMES,
        y=delta_net,
        text=np.round(delta_net, 2),
        textposition="outside",
        name="Delta NET"
    )
)

fig_delta_net.add_hline(y=0, line_width=1, line_color="black")

fig_delta_net.update_layout(
    title="Delta NET (Post - Pre)",
    xaxis_title="Variable",
    yaxis_title="Delta NET",
    height=500
)

fig_delta_net.show()

[PRE ] 2022-05-01 ~ 2022-10-31  -> 101 obs
[POST] 2023-01-01 ~ 2023-06-30 -> 98 obs

[TCI summary]
            period       start         end  n_obs   tci_mean   tci_std  \
0              pre  2022-05-01  2022-10-31  101.0  56.333637  7.712700   
1             post  2023-01-01  2023-06-30   98.0  53.862283  5.536769   
2  delta(post-pre)                            NaN  -2.471354       NaN   

     tci_min    tci_max  
0  44.001806  71.655635  
1  44.977571  67.676888  
2        NaN        NaN  

[NET mean summary]
      Variable  NET_mean_pre  NET_mean_post  Delta_NET
0  dlog_SOLVPN      3.066228      -3.132593  -6.198822
1  dlog_COPPER     -5.624142     -13.208975  -7.584833
2    dlog_GOLD      2.402389      14.502128  12.099738
3  dlog_SILVER      6.684336       7.991393   1.307057
4     dlog_DXY      0.217258       5.633493   5.416235
5     d_UST10Y     -7.476252      -6.064391   1.411861
6        d_VIX      0.730181      -5.721055  -6.451237

[Delta Connectedness: Post - Pre]
     